# MedGemma LoRA/QLoRA con dataset oficial

Este notebook entrena un adapter LoRA en modo cuantizado QLoRA para descripcion usando las transcripciones expertas del dataset actualizado.

Comparacion principal:

```text
MedGemma base BERTScore en test
vs
MedGemma + LoRA/QLoRA BERTScore en test
```


In [ ]:
from pathlib import Path

MODEL_ID = "google/medgemma-1.5-4b-it"
SPLIT_FILE = "dataset/split_repetition_1.json"
QLORA_OUTPUT_DIR = "checkpoints/official_medgemma_qlora_description"

## 1. Setup: Colab o cluster


In [ ]:
import os
import shutil
from pathlib import Path

try:
    import google.colab
    IS_COLAB = True
except ModuleNotFoundError:
    IS_COLAB = False

if IS_COLAB:
    !git clone https://github.com/Luco1421/utils_medgemma.git /content/utils_medgemma
    %cd /content/utils_medgemma/
    !pip install -q -r requirements.txt

## 2. Imports, token y dataset

In [ ]:
import json
import os
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import torch
from PIL import Image
from huggingface_hub import HfApi
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from bert_score import score as bertscore

HF_TOKEN = os.environ.get("HF_TOKEN")
if IS_COLAB and not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
DEVICE = "cuda:0"


In [ ]:
DATASET_ROOT = Path("dataset")
DESCRIPTION_PROMPT = "Describe the ophthalmological findings in this fundus image."

with open(SPLIT_FILE, "r", encoding="utf-8") as f:
    split_data = json.load(f)

def read_annotation(annotation_path):
    with open(DATASET_ROOT / annotation_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data[0] if isinstance(data, list) else data

def make_row(split_name, item):
    ann = read_annotation(item["annotation"])
    conditions = ann.get("locs_data", {}).get("conditions", []) or []
    return {
        "split": split_name,
        "image": str(DATASET_ROOT / item["image"]),
        "annotation": str(DATASET_ROOT / item["annotation"]),
        "label": ann.get("label"),
        "conditions": conditions,
        "target_label": "glaucoma" if "glaucoma" in [c.lower() for c in conditions] else "non_glaucoma",
        "prompt": DESCRIPTION_PROMPT,
        "answer": ann.get("transcription", ""),
        "reference": ann.get("transcription", ""),
        "locs_data": ann.get("locs_data", {}),
    }

rows = []
for split_name in ["train", "validation", "test"]:
    rows.extend(make_row(split_name, item) for item in split_data[split_name])

train_rows = [r for r in rows if r["split"] == "train" and r["answer"]]
val_rows = [r for r in rows if r["split"] == "validation" and r["answer"]]
test_rows = [r for r in rows if r["split"] == "test" and r["answer"]]

print("splits:", Counter(r["split"] for r in rows))
print("labels:", Counter(r["label"] for r in rows))
print("targets:", Counter(r["target_label"] for r in rows))
print("train/val/test:", len(train_rows), len(val_rows), len(test_rows))
train_rows[0]

## 3. Conversaciones multimodales para SFT

In [ ]:
def to_description_messages(row):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": Image.open(row["image"]).convert("RGB")},
                    {"type": "text", "text": row["prompt"]},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": row["answer"]}],
            },
        ]
    }

# Para smoke test rapido, puedes bajar TRAIN_LIMIT. Para entrenamiento completo, deja None.
TRAIN_LIMIT = None  # ejemplo: 12
train_source_rows = train_rows if TRAIN_LIMIT is None else train_rows[:TRAIN_LIMIT]
train_dataset = [to_description_messages(row) for row in train_source_rows]
print("training examples:", len(train_dataset))
train_dataset[0]

## 4. Cargar modelo 4-bit + LoRA

In [ ]:
major, minor = torch.cuda.get_device_capability()
TRAIN_DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("GPU capability:", (major, minor), "TRAIN_DTYPE:", TRAIN_DTYPE)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=TRAIN_DTYPE,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=TRAIN_DTYPE,
    device_map={"": 0},
    quantization_config=bnb_config,
    token=HF_TOKEN,
)

model.config.use_cache = False
try:
    model.gradient_checkpointing_enable()
except Exception as exc:
    print("gradient checkpointing no disponible:", exc)

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

## 5. Collate function

In [ ]:
def extract_image(messages):
    for msg in messages:
        content = msg.get("content", [])
        if not isinstance(content, list):
            content = [content]
        for element in content:
            if isinstance(element, dict) and element.get("type") == "image":
                return element["image"].convert("RGB")
    raise ValueError("No image found")

def collate_fn(examples):
    texts = []
    images = []
    for example in examples:
        messages = example["messages"]
        texts.append(processor.apply_chat_template(
            messages,
            add_generation_prompt=False,
            tokenize=False,
        ).strip())
        images.append(extract_image(messages))

    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    for token_name in ("boi_token_id", "image_token_id", "eoi_token_id"):
        token_id = getattr(processor.tokenizer, token_name, None)
        if token_id is not None:
            labels[labels == token_id] = -100

    batch["labels"] = labels
    return batch

## 6. Helpers de generación y BERTScore

In [ ]:
def generate_with_model(model_obj, processor_obj, prompt, image_path, max_new_tokens=384):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": Image.open(image_path).convert("RGB")},
            {"type": "text", "text": prompt},
        ],
    }]
    inputs = processor_obj.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    input_len = inputs["input_ids"].shape[-1]
    for key, value in inputs.items():
        if torch.is_tensor(value):
            if value.is_floating_point():
                inputs[key] = value.to(device=DEVICE, dtype=TRAIN_DTYPE)
            else:
                inputs[key] = value.to(device=DEVICE)
    with torch.inference_mode():
        output = model_obj.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output[0][input_len:]
    return processor_obj.decode(generated, skip_special_tokens=True).strip()

def evaluate_descriptions(outputs):
    candidates = [x["generated"] for x in outputs]
    references = [x["reference"] for x in outputs]
    P, R, F1 = bertscore(candidates, references, lang="en", rescale_with_baseline=True, verbose=True)
    for item, p, r, f in zip(outputs, P, R, F1):
        item["bertscore_precision"] = float(p)
        item["bertscore_recall"] = float(r)
        item["bertscore_f1"] = float(f)
    return {
        "count": len(outputs),
        "bertscore_precision_mean": float(P.mean()),
        "bertscore_recall_mean": float(R.mean()),
        "bertscore_f1_mean": float(F1.mean()),
    }, outputs

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

## 7. Baseline 4-bit antes de entrenar

In [ ]:
base_outputs = []
for idx, row in enumerate(test_rows, start=1):
    print(f"BASE [{idx}/{len(test_rows)}]", row["image"])
    generated = generate_with_model(model, processor, DESCRIPTION_PROMPT, row["image"], max_new_tokens=384)
    base_outputs.append({
        "image": row["image"],
        "label": row["label"],
        "conditions": row["conditions"],
        "reference": row["reference"],
        "generated": generated,
    })
    print(generated[:300])

base_summary, base_items = evaluate_descriptions(base_outputs)
base_summary

## 8. Entrenar QLoRA descriptivo

In [ ]:
training_args = SFTConfig(
    output_dir=QLORA_OUTPUT_DIR,
    max_steps=60,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=(TRAIN_DTYPE == torch.bfloat16),
    fp16=(TRAIN_DTYPE == torch.float16),
    logging_steps=5,
    save_steps=60,
    save_total_limit=1,
    remove_unused_columns=False,
    report_to="none",
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=processor,
    data_collator=collate_fn,
)

trainer.train()
trainer.save_model(QLORA_OUTPUT_DIR)
processor.save_pretrained(QLORA_OUTPUT_DIR)
model = trainer.model
model.eval()
print("Adapter guardado en:", QLORA_OUTPUT_DIR)

## 9. Evaluar QLoRA en test

In [ ]:
qlora_outputs = []
for idx, row in enumerate(test_rows, start=1):
    print(f"QLORA [{idx}/{len(test_rows)}]", row["image"])
    generated = generate_with_model(model, processor, DESCRIPTION_PROMPT, row["image"], max_new_tokens=384)
    qlora_outputs.append({
        "image": row["image"],
        "label": row["label"],
        "conditions": row["conditions"],
        "reference": row["reference"],
        "generated": generated,
    })
    print(generated[:300])

qlora_summary, qlora_items = evaluate_descriptions(qlora_outputs)
comparison = {
    "base": base_summary,
    "qlora": qlora_summary,
    "delta_f1": qlora_summary["bertscore_f1_mean"] - base_summary["bertscore_f1_mean"],
}
comparison

In [ ]:
save_json("results/official_description_base_vs_qlora_bertscore.json", {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "adapter_dir": QLORA_OUTPUT_DIR,
    "split": "test",
    "train_examples": len(train_dataset),
    "comparison": comparison,
    "base_results": base_items,
    "qlora_results": qlora_items,
})
print(json.dumps(comparison, indent=2))